# Operator actions and Symbolica export

This notebook walks through the **compiled Lagrangian** layer:

- **`Model` → `model.lagrangian()`** gives a `CompiledLagrangian` of ordered `InteractionTerm`s (the structure the vertex code trusts).
- **`L.to_symbolica()`** turns that into a single Symbolica expression for **display / scalar algebra** (factor order for fermions is **not** preserved in Symbolica).
- **`L.apply_operator(...)`** applies a graded Leibniz rule at the term level; optional **`flavor_expand`** matches `feynman_rules`.

Operators are configured with **`lagrangian.operator_action`** (small helpers like `replacement_operator`); everything else below uses the **model API** on the compiled object.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from symbolica import Expression, S

from model import Model, PartialD, dirac_field, flavor_index, scalar_field
from lagrangian.operator_action import FieldOperator, replacement_operator, single_field_result

## 1. Define fields and build the model

In [2]:
mu = S("mu")

Phi = scalar_field("Phi", self_conjugate=True)
Chi = scalar_field("Chi", self_conjugate=True)

model = Model(
    fields=(Phi, Chi),
    lagrangian_decl=Phi * Phi * Phi + Phi * PartialD(Phi, mu),
)

L = model.lagrangian()
print("Source (first declared piece):", model.lagrangian_decl.source_terms[0])
print("Compiled terms:", len(L.terms))

Source (first declared piece): Phi * Phi * Phi
Compiled terms: 2


## 2. Inspect compiled terms (ordered metadata)

Each term has `fields`, `derivatives`, `coupling`, and optional `closed_dirac_bilinears`. This is the authoritative ordering for fermions/ghosts.

In [3]:
def field_factor(occ):
    name = occ.field.name
    if occ.conjugated and not occ.field.self_conjugate:
        name = name + ".bar"
    return name


def describe_term(t, index):
    factors = " · ".join(field_factor(o) for o in t.fields)
    if t.derivatives:
        d = ", ".join(f"slot {a.target} → ∂_{a.lorentz_index}" for a in t.derivatives)
    else:
        d = "(none)"
    bil = t.closed_dirac_bilinears or "(none)"
    print(f"[{index}] {factors}")
    print(f"     ∂: {d}   bilinears: {bil}")


for i, term in enumerate(L.terms):
    describe_term(term, i)

[0] Phi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)


## 3. Symbolica export (`to_symbolica`)

Use this for **printing, simplifying coefficients, or quick checks**. Do not treat the product order in Symbolica as physical fermion order—that information stays in `L.terms`.

In [4]:
expr = L.to_symbolica()
print(expr)
print("Simplified:", Expression.parse(str(expr)).expand())

Phi*PartialD(Phi,mu)+Phi^3
Simplified: Phi*PartialD(Phi,mu)+Phi^3


## 4. Apply a field operator (`apply_operator`)

**Simple replacement:** map every `Phi` factor to `Chi` (graded Leibniz over the product).

**Product replacement + derivative:** if one slot becomes several fields and the slot carried `∂`, the engine **fans out** derivatives across the new slots (bosonic Leibniz), so gauge-like `φ → χ φ` on `φ ∂φ` is consistent.

In [5]:
replace_phi = replacement_operator("Phi_to_Chi", {Phi: Chi()})
L_replaced = L.apply_operator(replace_phi)

print("After replacing each Phi by Chi:", len(L_replaced.terms), "term(s)")
for i, term in enumerate(L_replaced.terms):
    describe_term(term, i)

print("\nSymbolica view:")
print(L_replaced.to_symbolica())

After replacing each Phi by Chi: 5 term(s)
[0] Chi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Chi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Phi · Chi
     ∂: (none)   bilinears: (none)
[3] Chi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[4] Phi · Chi
     ∂: slot 1 → ∂_mu   bilinears: (none)

Symbolica view:
Phi*PartialD(Chi,mu)+Chi*PartialD(Phi,mu)+3*Phi^2*Chi


In [6]:
split_phi = replacement_operator(
    "split_phi",
    {Phi: single_field_result((Chi(), Phi()))},
)
L_split = L.apply_operator(split_phi)

print("Product replacement on ∂ slots →", len(L_split.terms), "terms (derivative Leibniz).")
for i, term in enumerate(L_split.terms):
    describe_term(term, i)

Product replacement on ∂ slots → 6 terms (derivative Leibniz).
[0] Chi · Phi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Chi · Phi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Phi · Chi · Phi
     ∂: (none)   bilinears: (none)
[3] Chi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
[4] Phi · Chi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[5] Phi · Chi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)


## 5. Flavor-expanded export (`flavor_expand`)

Same keyword as for vertices: pass **`flavor_expand=True`** (or specific flavor indices) so **`to_symbolica`** serializes the **expanded** interaction list (class members like `e`, `mu`, `ta` instead of the generic `l`).

You can pass the same flag to **`apply_operator(..., flavor_expand=True)`** if you want the operator to act on that expanded view first.

In [7]:
Generation = flavor_index("Generation", 3, prefix="f")
lepton = dirac_field(
    "l",
    class_members=("e", "mu", "ta"),
    indices=(Generation,),
    flavor_index=Generation,
)
H = scalar_field("H", self_conjugate=True)
f = S("f")

flavor_model = Model(
    fields=(lepton, H),
    lagrangian_decl=S("g") * lepton.bar(f) * lepton(f) * H,
)
L_fl = flavor_model.lagrangian()

compact = L_fl.to_symbolica()
expanded = L_fl.to_symbolica(flavor_expand=True)

print("Generic (class field 'l'):")
print(compact)
print()
print("Flavor-expanded (e, mu, ta):")
print(expanded)

Generic (class field 'l'):
H*g*l(f,i_decl_1)*lbar(f,i_decl_1)

Flavor-expanded (e, mu, ta):
H*g*mu(i_decl_1)*mubar(i_decl_1)+H*g*ebar(i_decl_1)*e(i_decl_1)+H*g*tabar(i_decl_1)*ta(i_decl_1)


## 6. Optional: odd operator sign (fermion bookkeeping)

For **odd** operators (`parity=1`), the graded Leibniz rule inserts a **minus** when the operator crosses an odd number of fermionic factors to the left.

Compiled `psibar * psi` terms carry **`closed_dirac_bilinears`** metadata. If you replace a fermion inside that structure, the replacement must still expose exactly one matching **conjugated** (`psibar`) or **unconjugated** (`psi`) Dirac factor—here we map `psi → eta` while preserving that conjugation flag.

In [8]:
psi = dirac_field("psi", indices=())
eta = dirac_field("eta", indices=())

fermion_model = Model(
    fields=(psi, eta),
    lagrangian_decl=psi.bar() * psi(),
)
L_f = fermion_model.lagrangian()


def odd_s_on_psi(occurrence):
    if occurrence.field is not psi:
        return None
    # Preserve .bar vs plain so closed_dirac_bilinears stay valid.
    return single_field_result(eta.occurrence(conjugated=occurrence.conjugated))


s_odd = FieldOperator(name="s", parity=1, on_field=odd_s_on_psi)
L_s = L_f.apply_operator(s_odd)

print("Original:", len(L_f.terms), "term(s)")
for t in L_f.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))

print("\nAfter odd s on each psi slot:", len(L_s.terms), "terms")
for t in L_s.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))

Original: 1 term(s)
  coupling 1 → psi.bar · psi

After odd s on each psi slot: 2 terms
  coupling 1 → eta.bar · psi
  coupling -1 → psi.bar · eta


## 7. Scalar total-derivative investigation

Goal: investigate the minimal scalar identity

$$\partial_\mu(\phi^2\,\partial_\mu\phi)
= 2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi) + \phi^2\,\Box\phi$$

and therefore, **only modulo a total derivative / boundary term under the spacetime integral**, 

$$\phi^2\,\Box\phi \simeq -2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi).$$

In [9]:
c = S("c")
phi = Phi

current_model = Model(
    fields=(phi,),
    lagrangian_decl=phi * phi * PartialD(phi, mu),
)

expanded_divergence_model = Model(
    fields=(phi,),
    lagrangian_decl=(
        2 * phi * PartialD(phi, mu) * PartialD(phi, mu)
        + phi * phi * PartialD(PartialD(phi, mu), mu)
    ),
)

L1_model = Model(
    fields=(phi,),
    lagrangian_decl=c * phi * phi * PartialD(PartialD(phi, mu), mu),
)

L2_model = Model(
    fields=(phi,),
    lagrangian_decl=-2 * c * phi * PartialD(phi, mu) * PartialD(phi, mu),
)

delta_model = Model(
    fields=(phi,),
    lagrangian_decl=(
        c * phi * phi * PartialD(PartialD(phi, mu), mu)
        + 2 * c * phi * PartialD(phi, mu) * PartialD(phi, mu)
    ),
)

J = current_model.lagrangian()
J_expanded = expanded_divergence_model.lagrangian()
L1_td = L1_model.lagrangian()
L2_td = L2_model.lagrangian()
Delta_td = delta_model.lagrangian()

### 7.1 Current and present operator-layer status

The public DSL can represent the current $J^\mu = \phi^2\,\partial_\mu\phi$ directly.

What we **cannot** currently do naturally is ask the present `FieldOperator` replacement layer to generate the **outer** $\partial_\mu(\ldots)$ from scratch, because it replaces field slots and redistributes derivatives that already exist on slots; it does not create a fresh `DerivativeAction` on an otherwise underived product.

In [10]:
print("Current source:", current_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(J.terms):
    describe_term(term, i)

assert len(J.terms) == 1
assert [occ.field.name for occ in J.terms[0].fields] == ["Phi", "Phi", "Phi"]
assert [(a.target, str(a.lorentz_index)) for a in J.terms[0].derivatives] == [(2, "mu")]

print("Symbolica export:")
print(J.to_symbolica())

Current source: Phi * Phi * PartialD(Phi, mu)

[0] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
Symbolica export:
Phi^2*PartialD(Phi,mu)


### 7.2 Manual expansion of the divergence

Since the current operator-action layer does not yet create the fresh outer derivative, we build the right-hand side directly with the existing public `PartialD(...)` API and inspect the lowered terms.

In [11]:
print("Expanded source:", expanded_divergence_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(J_expanded.terms):
    describe_term(term, i)

assert len(J_expanded.terms) == 2
assert J_expanded.terms[0].coupling == 2
assert J_expanded.terms[1].coupling == 1
assert [(a.target, str(a.lorentz_index)) for a in J_expanded.terms[0].derivatives] == [(1, "mu"), (2, "mu")]
assert [(a.target, str(a.lorentz_index)) for a in J_expanded.terms[1].derivatives] == [(2, "mu"), (2, "mu")]

print("Symbolica export:")
print(J_expanded.to_symbolica())
print("Expanded:")
print(Expression.parse(str(J_expanded.to_symbolica())).expand())

Expanded source: 2 * Phi * PartialD(Phi, mu) * PartialD(Phi, mu)

[0] Phi · Phi · Phi
     ∂: slot 1 → ∂_mu, slot 2 → ∂_mu   bilinears: (none)
[1] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu, slot 2 → ∂_mu   bilinears: (none)
Symbolica export:
2*Phi*PartialD(Phi,mu)^2+Phi^2*PartialD(PartialD(Phi,mu),mu)
Expanded:
2*Phi*PartialD(Phi,mu)^2+Phi^2*PartialD(PartialD(Phi,mu),mu)


### 7.3 Compare $L_1$, $L_2$, and $L_1 - L_2$

Define

$$L_1 = c\,\phi^2\,\Box\phi, \qquad L_2 = -2c\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi).$$

Then

$$L_1 - L_2 = c\,\partial_\mu(\phi^2\,\partial_\mu\phi).$$

So the two Lagrangians are equivalent **modulo a total derivative**, not as identical local expressions.

In [12]:
expr_J_expanded = J_expanded.to_symbolica()
expr_L1 = L1_td.to_symbolica()
expr_L2 = L2_td.to_symbolica()
expr_Delta = Delta_td.to_symbolica()
expr_cJ = c * expr_J_expanded
expr_delta_check = (expr_Delta - expr_cJ).expand()

print("L1 =")
print(expr_L1)
print()
print("L2 =")
print(expr_L2)
print()
print("L1 - L2 =")
print(expr_Delta)
print()
print("c * expanded divergence =")
print(expr_cJ)
print()
print("Expanded check:")
print(expr_delta_check)

assert expr_delta_check.to_canonical_string() == "0"
print()
print("Check passed: L1 - L2 matches c * ∂_μ(φ² ∂_μ φ) after manual expansion.")


L1 =
Phi^2*c*PartialD(PartialD(Phi,mu),mu)

L2 =
-2*Phi*c*PartialD(Phi,mu)^2

L1 - L2 =
2*Phi*c*PartialD(Phi,mu)^2+Phi^2*c*PartialD(PartialD(Phi,mu),mu)

c * expanded divergence =
c*(2*Phi*PartialD(Phi,mu)^2+Phi^2*PartialD(PartialD(Phi,mu),mu))

Expanded check:
0

Check passed: L1 - L2 matches c * ∂_μ(φ² ∂_μ φ) after manual expansion.


### 7.4 Conclusion

- The framework can represent both sides of this scalar identity with the current public declarative API.
- The present `FieldOperator` layer is not yet the right abstraction for creating a fresh outer spacetime derivative.
- `to_symbolica()` is useful here for scalar display and algebra checks **after** lowering.
- Symbolica does **not** know the physics statement that a total derivative only contributes a boundary term under $\int d^4x$.
- A future automatic IBP / total-derivative simplification layer should sit above the lowered `InteractionTerm` representation, and should stay separate from `vertex_engine.py` and likely also separate from the current replacement-style operator engine.

## 8. Ordinary partial derivative as a linear operator: current limitation

This section corrects an important distinction. A replacement like `Phi -> Chi` is a field-space variation. It is **not** the spacetime derivative. The ordinary derivative operator should satisfy

$$\partial_\mu(\Phi^3) = 3\Phi^2\,\partial_\mu\Phi.$$

So the natural slot action would be `d_mu[Phi] = PartialD(Phi, mu)`.


In [13]:
lam = S("lam")

cubic_partial_model = Model(fields=(Phi,), lagrangian_decl=lam * Phi * Phi * Phi)
expected_dmu_model = Model(
    fields=(Phi,),
    lagrangian_decl=3 * lam * Phi * Phi * PartialD(Phi, mu),
)
L_cubic_partial = cubic_partial_model.lagrangian()
expected_dmu = expected_dmu_model.lagrangian()
expr_expected_dmu = expected_dmu.to_symbolica()

print("Source:", cubic_partial_model.lagrangian_decl.source_terms[0])
print("Expected target:", expected_dmu_model.lagrangian_decl.source_terms[0])
print()
print("Expected lowered output if d_mu acted by the product rule:")
for i, term in enumerate(expected_dmu.terms):
    print("coupling", term.coupling)
    describe_term(term, i)
print()
print("Expected Symbolica export:")
print(expr_expected_dmu)

assert len(expected_dmu.terms) == 1
assert expr_expected_dmu.expand().to_canonical_string() == Expression.parse(
    "3*lam*Phi^2*PartialD(Phi,mu)"
).to_canonical_string()


Source: lam * Phi * Phi * Phi
Expected target: 3*lam * Phi * Phi * PartialD(Phi, mu)

Expected lowered output if d_mu acted by the product rule:
coupling 3*lam
[0] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)

Expected Symbolica export:
3*Phi^2*lam*PartialD(Phi,mu)


### 8.1 Attempt with the current operator-action API

The present `FieldOperator` layer stores operator outputs as ordered tuples of `FieldOccurrence`s plus a scalar coefficient. Fresh derivatives live separately in `InteractionTerm.derivatives`.


In [14]:
replacement_error = None
custom_export_error = None
invalid_field_types = None

try:
    d_mu_replacement = replacement_operator("d_mu", {Phi: PartialD(Phi, mu)})
    attempted_from_mapping = L_cubic_partial.apply_operator(d_mu_replacement)
except Exception as err:
    replacement_error = err

try:
    def on_field_partial(occ):
        if occ.field is Phi:
            return single_field_result((PartialD(Phi, mu),))
        return None

    d_mu_custom = FieldOperator(name="d_mu", parity=0, on_field=on_field_partial)
    attempted_dmu = L_cubic_partial.apply_operator(d_mu_custom)
    invalid_field_types = [type(f).__name__ for f in attempted_dmu.terms[0].fields]
    attempted_dmu.to_symbolica()
except Exception as err:
    custom_export_error = err

print("replacement_operator attempt:")
print(f"{type(replacement_error).__name__}: {replacement_error}")
print()
print("custom FieldOperator attempt:")
print("field entry types in the first transformed term:", invalid_field_types)
print(f"{type(custom_export_error).__name__}: {custom_export_error}")

assert isinstance(replacement_error, TypeError)
assert "FieldOccurrence or OperatorAtomResult" in str(replacement_error)
assert invalid_field_types == ["PartialDerivativeFactor", "FieldOccurrence", "FieldOccurrence"]
assert isinstance(custom_export_error, AttributeError)
assert "species" in str(custom_export_error)



replacement_operator attempt:
TypeError: replacement_operator('d_mu'): mapping values must be FieldOccurrence or OperatorAtomResult, got PartialDerivativeFactor.

custom FieldOperator attempt:
field entry types in the first transformed term: ['PartialDerivativeFactor', 'FieldOccurrence', 'FieldOccurrence']
AttributeError: 'PartialDerivativeFactor' object has no attribute 'species'


### 8.2 Conclusion

- The framework can represent the target expression `3 lam Phi^2 PartialD(Phi, mu)` once it is written declaratively.
- The current `FieldOperator` API cannot yet encode `d_mu[Phi] = PartialD(Phi, mu)`.
- The reason is structural: operator replacements are `FieldOccurrence` tuples, while a fresh spacetime derivative must be added as new `DerivativeAction` metadata on a slot.
- So the earlier `Phi -> Chi` examples should be interpreted as variation-like replacement operators, not as an ordinary spacetime derivative.
- A future true `d_mu` operator would need a richer operator result type that can create new derivative annotations on the replacement slots.
